# 4. Parser performance

*Author: La Anh Hào · code: `src/15_run_parsing.ipynb`*

The statement asks for the running time of the parser (end-to-end wall time and average time per paper) and
for its memory footprint (maximum RAM and the size of the final output), measured on Google Colab in CPU-only
mode. `src/15_run_parsing.ipynb` measures them itself while it parses: every session writes a record with its
wall time, the papers it parsed and its RAM, sampled every 2 seconds for the notebook process **and all its
worker processes**; the record is saved every few minutes, so a session that Colab stops still counts.
`m2_benchmark.json` sums the sessions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import report_data as rd

rd.check()
rd.style()
bench, stats = rd.benchmark(), rd.stats()
sessions, versions, papers = rd.sessions(), rd.versions(), rd.papers()
main = sessions.loc[sessions["papers"].idxmax()]   # the session that parsed the most papers
parsers = sessions["parser_version"].str.split("|").str[0]
first = sessions[parsers == parsers.iloc[0]]        # the sessions of the first parser version
full_pass = int(first["papers"].sum()) >= int(stats["papers"])
later = sessions[parsers != parsers.iloc[0]]
pass_wall = first["wall_seconds"].sum() if full_pass else bench["wall_seconds"]
hm = rd.hm

## 4.1 Testbed and sessions

`15_run_parsing` runs `parse_paper` (chapter [2](hierarchy.md)) in a pool of worker processes, one paper per
task, on a Colab CPU runtime with the corpus on Google Drive. Reading the source files from Drive, not the
CPU, limits the speed, so the pool has more workers than the runtime has cores. A session parses the papers
that the resume index does not list yet and builds the three Parquet tables at the end.

In [ ]:
t41 = pd.DataFrame({
    "Start (UTC)": sessions["start"].dt.strftime("%Y-%m-%d %H:%M"),
    "Wall time": [hm(s) for s in sessions["wall_seconds"]],
    "Papers parsed": sessions["papers"],
    "Versions": sessions["versions"],
    "Workers": sessions["workers"],
    "Peak RAM (MB)": sessions["ram_peak_mb"],
    "Parser": sessions["parser_version"].str.split("|").str[0],
    "Status": np.where(sessions["complete"], "completed", "interrupted"),
})
rd.table(t41, "Table 4.1 – Sessions of 15_run_parsing", {"Peak RAM (MB)": "{:,.0f}"})
text = (f"Runtime: `{bench.get('colab_cpu', '?')}`, **{bench.get('workers', '?')}** worker processes. "
        f"The main session ({main['start']:%d %b}) parsed **{main['papers']:,}** papers in "
        f"**{hm(main['wall_seconds'])}**")
if full_pass and len(first) > 1:
    n_before = len(first) - 1
    text += (f"; with the {'session' if n_before == 1 else f'{n_before} sessions'} before it, parser "
             f"{parsers.iloc[0]} parsed all "
             f"**{int(first['papers'].sum()):,}** papers in **{hm(pass_wall)}**")
text += "."
if len(later):
    text += (f" The last session ran parser {parsers.iloc[-1]}: it parsed again only the "
             f"**{int(later['papers'].sum()):,}** papers that the change from {parsers.iloc[0]} could affect and kept "
             f"the results of the others (chapter [2.5](running.md)).")
rd.md(text)

## 4.2 Running time

**End-to-end wall time** is the time between the start and the end of the sessions, setup and the
construction of the tables included. The **average time per paper** divides it by the number of papers: it is
the throughput of the whole pipeline, with its parallel workers. The time a worker spends on one paper
(reading its files, parsing every version, writing `data/parsed/<id>.json`) is recorded too; it is longer
than the average because several workers run at once.

In [ ]:
worker = papers["seconds"].dropna()
worker = worker[worker > 0]
rows = []
if full_pass:
    rows += [(f"End-to-end wall time of one full pass (parser {parsers.iloc[0]})",
              f"{hm(pass_wall)} for {int(first['papers'].sum()):,} papers"),
             ("Average time per paper, full pass", f"{pass_wall / int(first['papers'].sum()):.3f} s")]
for row in later.itertuples():
    rows.append((f"Session of parser {row.parser_version.split('|')[0]} (tables of all papers rebuilt)",
                 f"{hm(row.wall_seconds)} for {row.papers:,} papers"))
rows += [
    ("End-to-end wall time, all sessions", hm(bench["wall_seconds"])),
    (f"Average time per paper, all sessions ({hm(bench['wall_seconds'])} / {bench['papers']:,} papers)",
     f"{bench['avg_seconds_per_paper']:.3f} s"),
    ("Time per paper in a worker: median / 90th percentile / maximum",
     f"{worker.median():.2f} s / {worker.quantile(0.9):.2f} s / {worker.max():,.1f} s"),
]
rd.table(pd.DataFrame(rows, columns=["Metric", "Value"]), "Table 4.2 – Running time")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
ax = axes[0]
bins = np.logspace(np.log10(worker.min()), np.log10(worker.max()) + 0.02, 60)
ax.hist(worker, bins=bins, color=rd.BLUE, edgecolor=rd.SURFACE, linewidth=0.5)
ax.set_xscale("log")
ax.axvline(worker.median(), color=rd.ORANGE, lw=1.5)
ax.set_xlabel("Seconds in a worker (log scale)")
ax.set_ylabel("Papers")
ax.grid(axis="x", visible=False)
ax.set_title("Time per paper")
ax = axes[1]
sized = papers[(papers["seconds"] > 0) & (papers["nodes"] > 0)]
ax.scatter(sized["nodes"], sized["seconds"], s=3, alpha=0.25, color=rd.BLUE, linewidths=0)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Tree nodes of the paper, all versions (log)")
ax.set_ylabel("Seconds in a worker (log)")
ax.set_title("Time against size")
plt.tight_layout()
plt.show()
rd.figure_caption("Figure 4.1 – Left: time a worker spent on each paper (orange line: median). Right: the same "
                  "time against the number of tree nodes of the paper.")
slow = papers.nlargest(8, "seconds")[["arxiv_id", "versions", "nodes", "bib_before", "seconds"]]
rd.table(slow.rename(columns={"arxiv_id": "Paper", "versions": "Versions", "nodes": "Tree nodes",
                              "bib_before": "BibTeX entries", "seconds": "Seconds"}),
         "Table 4.3 – The slowest papers", {"Seconds": "{:,.1f}"})

In [ ]:
rho = sized["nodes"].rank().corr(sized["seconds"].rank())   # Spearman: Pearson of the ranks
timeouts = int(versions["error"].str.startswith("timeout").sum())
rd.md(f"The time grows with the size of the paper (Spearman correlation between tree nodes and seconds: "
      f"**{rho:.2f}**). `15` stops a paper after `PAPER_TIMEOUT` = 600 s and records its versions as errors "
      f"instead of blocking the run; "
      + ("no paper reached that limit." if timeouts == 0 else f"{timeouts:,} versions reached that limit."))

## 4.3 Memory footprint

In [ ]:
series = pd.DataFrame(bench.get("ram_timeseries", []), columns=["seconds", "mb"])
if len(series):
    gap = series["seconds"].diff().fillna(0)
    counts = sessions["ram_samples"] if "ram_samples" in sessions.columns else None
    if counts is not None and counts.notna().all() and int(counts.sum()) == len(series):
        starts = [0] + list(np.cumsum(counts.astype(int)))          # the samples of each session, in order
    else:   # older logs: a pause of more than 10 min without a sample starts a new session
        starts = [0] + list(np.where(gap > 600)[0]) + [len(series)]
    pause = pd.Series(0.0, index=series.index)
    for a in starts[1:-1]:
        pause.iloc[a] = gap.iloc[a] - 2          # the time between two sessions, removed from the axis
    series["hours"] = (series["seconds"] - pause.cumsum()) / 3600
    fig, ax = plt.subplots(figsize=(9.5, 3.2))
    for a, b in zip(starts[:-1], starts[1:]):
        part = series.iloc[a:b]
        ax.plot(part["hours"], part["mb"], color=rd.BLUE, lw=1.2)
        if a:
            ax.axvline(part["hours"].iloc[0], color=rd.AXIS, lw=1, ls="--")
    ax.axhline(bench["peak_ram_mb"], color=rd.ORANGE, lw=1, ls=":")
    ax.text(series["hours"].max(), bench["peak_ram_mb"], f" peak {bench['peak_ram_mb']:,.0f} MB", va="bottom",
            ha="right", color=rd.ORANGE, fontsize=9)
    ax.set_ylim(0, bench["peak_ram_mb"] * 1.15)
    ax.set_xlabel("Hours of parsing (sessions one after the other, dashed line: a new session)")
    ax.set_ylabel("RAM of the parser (MB)")
    ax.set_title("RAM during the run")
    plt.tight_layout()
    plt.show()
    rd.figure_caption("Figure 4.2 – Resident memory of the notebook process and its workers, sampled every 2 s.")
rd.md(f"**Peak RAM: {bench['peak_ram_mb']:,.0f} MB**, time-weighted average **{bench['avg_ram_mb']:,.0f} MB**, "
      f"for the notebook and its {bench.get('workers', '?')} workers together. The value is the sum of the resident "
      f"memory of the processes: the workers are forked from the notebook and share some of its memory pages, which "
      f"the sum counts once per process, so it is an upper bound of the memory used. The memory does not grow with the "
      f"number of papers parsed: each worker holds one paper at a time and the per-paper caches are emptied after "
      f"it, and the tables are written to Parquet in batches of 500 papers instead of being built in memory.")

## 4.4 Output size

The output has two parts: `data/parsed/`, one JSON file per paper with the trees of all its versions and its
rows (the input of Milestone 3, enough to rebuild the tables without parsing again), and the three Parquet
tables of the statement in `data/m2/`.

In [ ]:
last = sessions.iloc[-1]
total = bench["output_bytes"]
parquet = last["parquet_bytes"] if "parquet_bytes" in sessions.columns and pd.notna(last["parquet_bytes"]) else None
parsed_n = int(stats["papers_parsed"])
rows = [("data/parsed/ (one JSON file per paper)", None if parquet is None else (total - parquet) / rd.MB),
        ("data/m2/*.parquet (elements, hierarchy, bibtex)", None if parquet is None else parquet / rd.MB),
        ("Total", total / rd.MB)]
t44 = pd.DataFrame(rows, columns=["Output", "Size (MB)"])
rd.table(t44, "Table 4.4 – Size of the output", {"Size (MB)": "{:,.1f}"})
t45 = pd.DataFrame([("elements", int(papers["elements"].sum())), ("hierarchy", int(papers["hierarchy"].sum())),
                    ("bibtex", int(papers["bib_after"].sum()))], columns=["Table", "Rows"])
rd.table(t45, "Table 4.5 – Rows of the three tables")
rd.md(f"The output takes **{total / rd.GB:,.2f} GB**, **{total / max(parsed_n, 1) / 1024:,.0f} KB** per paper on "
      f"average. Most of it is `data/parsed/`, which repeats the rows with the tree of every version; the Parquet "
      f"tables are compressed with zstd and store the element `kind` as a dictionary column.")

## 4.5 Summary of the performance metrics

In [ ]:
pass_papers = int(first["papers"].sum()) if full_pass else bench["papers"]
rows = [
    ("Running time", "End-to-end wall time" + (" (one full pass)" if full_pass and len(later) else ""), hm(pass_wall)),
    ("Running time", "Average time per paper", f"{pass_wall / pass_papers:.3f} s"),
    ("Running time", "Papers / versions parsed", f"{bench['papers']:,} / {bench['versions']:,}"),
    ("Memory", "Maximum RAM", f"{bench['peak_ram_mb']:,.0f} MB"),
    ("Memory", "Average RAM", f"{bench['avg_ram_mb']:,.0f} MB"),
    ("Memory", "Final output size", f"{bench['output_bytes'] / rd.GB:,.2f} GB"),
    ("Testbed", "Runtime", f"Google Colab, CPU only: {bench.get('colab_cpu', '?')}"),
    ("Testbed", "Worker processes", f"{bench.get('workers', '?')}"),
]
rd.table(pd.DataFrame(rows, columns=["Group", "Metric", "Value"]), "Table 4.6 – Performance metrics required by the statement")